# Paper Story Board
- FDT/HMI cross calibration available but not easily applicable yet
- Synoptic map data set features periods where SDO is close to Earth and PHI produces maps that are almost identical in structure to the HMI maps
- direct pixel-to-pixel comparison in scatter plots as typical for cross calibration problems not applicable since to strict co-observation is available
  - observations are hours to about a day apart from the PHI orbit location
  - 2D histogram serves as example how no good correlation exists for this kind of analysis
  - 
- alternative way to assess the quality of combined synoptic maps necessary
  - flux balance and unsigned flux hint at the necessity of a cross calibration and somewhat confirm Alejandro et al
  - zero level analysis shows sub Gauss offsets, which are not enough to fix the flux balance
  - tinkering showed about 0.5 G necessary to balance wrt to HMI
  - flux balance apparently by applying the offset of Alejandro's cross calibration fit to filtered AR regions (at 250G seeds, 25 lower thld I think)
  - AR filtered pixels are about 10% of the total amount which at 5 G offset is about the 0.5 G found in above tinkering 
  - conclusion: cross calibration offset seems to provide the flux balance
  - total flux possibly corrected with cross calibration scaling but filtering for flux needs improvement
- AR flux filter
  - currently based on reconstruction from high seed to low threshold
  - try iterative version where we seed eg from 250->25, 200->25, 150->25 and combine all masks into a single mask that hopefully also selects for the lower overall magnetic activiy in PHI
  - check Burkhart Bovelet papers for multi level method used for bright points
  
- use 2290, 2291, 2293, 2294 in front of Earth and hopefully produce almost identical PFSS results
  - if PFSS is identical despite above quantification of the possible error, we can consider the PFSS results robust within error bars and discuss morphological differences in the farside observations
  - figure out if lack of decaying active region bananas in PHI makes a difference for PFSS (use differnet filtering)

- morphological differences in farside observations
  - possibly quantify with the geographical differences in magnetic activity
  - this can hopefully be done through the AR filter masks

## ToDo List

- see how using blos affects things, I'm wondering if the µ multiplication is messing with the results
  - confirm how our blos data was created (inversion?)

- check flux distribution and balance after binning
  - compare with convolved and non convolved binned HMI 
- find a better temporal average for the distance
- repeat for the other 3 synptic maps at that area

- Pipeline
  - try constructing the synoptic map at 1800x720px and maybe even 960x480px resolution
  - synoptic map module needs adjustments to mapmmax and sinbdivs parameters
  - the deg per pixel are probably hard coded for the weight functions so this needs dynamic calculation
    - potential to break AWF if we're not careful with the decimals here
  -  sinbdivs, mapmmax and rebin factor are currently hard coded in m720s_drms_pipe and phi_drms_interface
     -  above config parameter can possibly be used but gotta be careful with unrebinned dimensions in these modules

  -  Manually degrade a combined synoptic map and check if the PFSS makes any difference before changing all of this
  -  
## Take aways
  - results generally improve with aggressive blur
  

# Code

## Imports

In [ ]:
import os, sys, glob
import pandas as pd
import numpy as np
from scipy import optimize
from astropy.io import fits
import drms
from matplotlib import pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path

from matplotlib.colors import LogNorm
from skimage.morphology import reconstruction
from scipy import ndimage as ndi

from scipy.signal import fftconvolve
from astropy.convolution import convolve
from scipy.special import j1


In [ ]:
SRC_PATH = (Path.cwd() / "../src").resolve()
sys.path.insert(0, str(SRC_PATH))

In [ ]:
from config.config import Config
from utils.plots import magnetic_flux_plot_latitudes, combined_synoptic_noise_plot, plot_synoptic_sources, plot_synoptic_with_stripe_magnitudes

In [ ]:
%matplotlib widget

## Function definitions

In [ ]:

def ar_filtering(img, high_thld=250.0, low_thld=25.0, empty=0.0):

    from skimage.morphology import reconstruction

    seed = np.abs(img) >= high_thld
    mask = np.abs(img) >= low_thld

    final_mask = reconstruction(
        seed.astype(np.uint8),
        mask.astype(np.uint8),
        method='dilation'
    ).astype(bool)
    filtered_img = np.where(final_mask, img, empty)

    return filtered_img, final_mask

In [ ]:
def ar_filtering_with_labels(img, high_thld, low_thld):
    
    from skimage.morphology import reconstruction
    from scipy import ndimage as ndi

    seed = np.abs(img) >= high_thld
    mask = np.abs(img) >= low_thld

    final_mask = reconstruction(
        seed.astype(np.uint8),
        mask.astype(np.uint8),
        method='dilation'
    ).astype(bool)

    labels, n_regions = ndi.label(final_mask)

    return labels, final_mask, n_regions

In [ ]:
def extract_cr(name: str) -> int:
    # name like "CR1234" or "CR1234_PHI"
    base = name.split("_")[0]    # → "CR1234"
    return int(base[2:])          # extract number

In [ ]:
def label_active_regions(img, high_thld, low_thld, step):

    ar_img_list      = []
    ar_labels_list   = []
    ar_mask_list     = []
    ar_nregions_list = []

    remaining_img = img.copy()
    remaining_img_list = []

    for thld in np.linspace(high_thld, low_thld, int(high_thld//step)):

        high_thld = thld

        ar_labels, ar_mask, n_regions = ar_filtering_with_labels(remaining_img, high_thld, low_thld)

        ar_img  = np.where(ar_mask, img, np.nan)
        remaining_img = np.where(~ar_mask, remaining_img, np.nan)

        ar_img_list.append(ar_img)
        ar_labels_list.append(ar_labels)
        ar_mask_list.append(ar_mask)    
        ar_nregions_list.append(n_regions)
        remaining_img_list.append(remaining_img)
    
    
    # consolidate ar_labels_list into single label map
    ar_labels = ar_labels_list[0]

    for imask in range(len(ar_labels_list[1:])):

        tmp_mask   = np.where(ar_labels_list[imask+1] > 0, True, False)
        tmp_labels = np.where(tmp_mask, ar_labels_list[imask+1] + np.max(ar_labels), 0)

        ar_labels = ar_labels + tmp_labels

    ar_labels  = np.where(ar_labels > 0, ar_labels, np.nan)

    img_mask = ar_labels > 0
    ar_img = np.where(img_mask, img, np.nan)

    return ar_img, ar_labels, img_mask #ar_img_list, remaining_img_list, ar_labels_list, ar_mask_list, ar_nregions_list

## File I/O

In [ ]:
# Complete CR 2280, 2283, 2284, 2285, 2286, 2287, 2291, 2293, 2294, 2296, 2298, 2299, 2300
# 2290, 2291, 2293, 2293 in front of Earth view
# HDIS: 
# 2290 = 2024.10.21_00:30:03_TAI-2024.11.12_18:15:03_TAI = ~0.6 AU
# 2291 = 2024.11.12_21:15:03_TAI-2025.01.05_15:15:09_TAI = ~0.8 AU
# 2293 = 2025.01.05_18:15:09_TAI-2025.02.01_12:15:09_TAI = ~0.8 AU
# 2294 = 2025.02.01_15:45:09_TAI-2025.03.01_06:15:09_TAI = ~0.7 AU 
carrington_number = 2291

In [ ]:
root = Path.cwd()
#path = Path('/scratch/slam/loeschl/dev/python/synoptic-map-pipeline/output/release_2025_v01/l3/syn/PHIHMI/CR%s/synop/'%carrington_number)
path = Path('/scratch/slam/loeschl/dev/python/synoptic-map-pipeline/output/release_2025_v01/l3/syn/PHI/CR%s_PHI/synop/'%carrington_number)

fname = "synopMr.fits"
series="hmi.synoptic_mr_polfil_720s"
segment="Mr_polfil"
small = False

#fname   = "synopMr_small.fits"
#series  = "hmi.mrsynop_small_720s"
#segment = "synopMr"
#small   = True


#path = Path('/scratch/slam/loeschl/dev/python/synoptic-map-pipeline/output/CR2291_PHI_only_blos/synop/')
#fname = "synopMl.fits"
#series="hmi.synoptic_ml_720s"
#segment="synopMl"
#small = False

In [ ]:
############################
####### GET PHI DATA #######
############################

synop_phi  = fits.open(path / fname)

phi_img   = synop_phi[0].data
if not small: 
    phi_table = synop_phi[1].data

# Define Carrington rotation number
#carrington_number = int(synop_phi[0].header["CAR_ROT"])
outname = f"{carrington_number}"


############################
####### GET HMI DATA #######
############################

# Create DRMS client
c = drms.Client(email="loeschl@mps.mpg.de", verbose=True)

datapath_hmi = os.path.join(root, f"../data/tmp/CR{carrington_number}/")
os.makedirs(datapath_hmi, exist_ok=True)

# Query JSOC for that rotation
q = c.query(f"{series}[{carrington_number}]", seg=segment)
fname_hmi = q[segment][0].split('/')[-1]

try:
    file_hmi  = glob.glob(f"{datapath_hmi}*{fname_hmi}")[0]
    synop_hmi = fits.open(file_hmi)
except IndexError:
    # Download the FITS file
    result = c.export(f"{series}[{carrington_number}]", method='url', protocol='fits')
    result.download(datapath_hmi)
    file_hmi  = glob.glob(f"{datapath_hmi}*{fname_hmi}")[0]
    synop_hmi = fits.open(file_hmi)

try: 
    hmi_img = synop_hmi[1].data   

    # create mask of NaN values in phi_img and fill them with hmi_img values
    mask_polfil = np.isnan(phi_img)
    phi_polfil  = np.where(mask_polfil, hmi_img, phi_img)    

    # aggressive HMI pole filling
    #phi_polfil[:40, :]  = hmi_img[:40, :]
    #phi_polfil[1400: :] = hmi_img[1400:, :]

    #synop_phi[0].header["CUNIT2"] = "deg" # "Sine Latitude"

    primary_hdu = fits.PrimaryHDU()

    polfil_hdu = fits.CompImageHDU(data=phi_polfil, 
                                    header=synop_phi[0].header, 
                                    compression_type='RICE_1')

    polfil_hdul = fits.HDUList([primary_hdu, polfil_hdu])
    polfil_hdul.writeto(os.path.join(path, "synopMr_polfil.fits"), overwrite=True)

    # show filled synoptic map 
    #plt.imshow(phi_polfil, cmap='hmimag', vmin=-1500, vmax=1500, origin='lower')
    #plt.show()

except IndexError:
    # non polfil data
    hmi_img = synop_hmi[0].data

# populate config for diagnostics plots
config = Config(path / "../config.yaml")
#config.cr = carrington_number
config.update("cr", carrington_number)

if "Mr" in segment:
    config.Mr = True
    config.Btype = "Radial"
else:
    config.Mr = False
    config.Btype = "line-of-sight"

# Analysis

In [ ]:
high_thld = 250
low_thld  = 25
step = 25

## Overview

Obvious difference in resolution between HMI and PHI. State of magnetic field mostly identical

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_img, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')
im2 = ax[1].imshow(hmi_img, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI CR')

ax[0].set_title('PHI CR %s'%carrington_number)
ax[1].set_title('HMI CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

## Active Region and Background Separation

In [ ]:
phi_ar_img, phi_ar_labels, phi_ar_mask = label_active_regions(phi_img, high_thld, low_thld, step)
hmi_ar_img, hmi_ar_labels, hmi_ar_mask = label_active_regions(hmi_img, high_thld, low_thld, step)

phi_rest = np.where(~phi_ar_mask, phi_img, np.nan)


In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar_img,   vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(phi_rest, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

Top: Active region with seed pixels > thld_high are selected and dilated down to thld_low. Bottom: remaining pixel with field strenghts < thld_low

## Active Regions in PHI and HMI Maps

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar_img, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_ar_img, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

In [ ]:
hmi_pos = hmi_ar_img[hmi_ar_img>=0].flatten().sum()
hmi_neg = hmi_ar_img[hmi_ar_img<0].flatten().sum()

phi_pos = phi_ar_img[phi_ar_img>=0].flatten().sum()
phi_neg = phi_ar_img[phi_ar_img<0].flatten().sum()    

hmi_ar_unsigned = np.nansum(np.abs(hmi_ar_img).flatten())
phi_ar_unsigned = np.nansum(np.abs(phi_ar_img).flatten())


In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative', 'HMI unsigned', 'PHI unsigned'], [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_ar_unsigned, phi_ar_unsigned])
ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map AR") 

More HMI flux from more active active pixels in the hmi_ar_mask? PHI not flux balanced and inverted wrt to HMI flux balance

In [ ]:
ratio_unsigned_flux = hmi_ar_unsigned/phi_ar_unsigned
print(f'Unsigned flux HMI {hmi_ar_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux:.4f}')



## Combined AR Mask

Combine active pixels from both masks in an attempt to increase unsigned PHI flux

- overview plot
- flux plot

In [ ]:
n_phi_ar_img = len(phi_ar_img[~np.isnan(phi_ar_img)])
n_hmi_ar_img = len(hmi_ar_img[~np.isnan(hmi_ar_img)])
n_ar_img_ratio  = n_phi_ar_img/n_hmi_ar_img

print(f"N pixel in HMI mask: {n_phi_ar_img}")
print(f"N pixel in PHI mask: {n_hmi_ar_img}")
print(f"N pixel ratio in PHI/HMI masks: {n_ar_img_ratio:.4f}")

In [ ]:
# build combined mask from HMI and PHI AR masks
hmi_ar_mask = ~np.isnan(hmi_ar_img)
phi_ar_mask = ~np.isnan(phi_ar_img)

# create master mask from phi_ar_mask and hmi_ar_mask
ar_mask = hmi_ar_mask | phi_ar_mask

len(hmi_ar_mask[hmi_ar_mask==True]), len(phi_ar_mask[phi_ar_mask==True]), len(hmi_ar_mask[ar_mask==True])

In [ ]:
phi_ar_comb  = np.where(ar_mask, phi_img, np.nan)
hmi_ar_comb  = np.where(ar_mask, hmi_img, np.nan)

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 10), nrows=3, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar_img, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(phi_ar_comb, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='PHI')
im3 = ax[2].imshow(hmi_ar_comb, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='HMI')

ax[0].set_title(f'PHI Active Regions CR {carrington_number}')
ax[1].set_title(f'PHI Active Regions CR {carrington_number} - Combined Mask')
ax[2].set_title(f'HMI Active Regions CR {carrington_number} - Combined Mask')

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')
ax[2].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')
fig.colorbar(im3, label='Br [G]')


In [ ]:

hmi_ar_comb_pos = hmi_ar_comb[hmi_ar_comb>=0].flatten().sum()
hmi_ar_comb_neg = hmi_ar_comb[hmi_ar_comb<0].flatten().sum()

phi_ar_comb_pos = phi_ar_comb[phi_ar_comb>=0].flatten().sum()
phi_ar_comb_neg = phi_ar_comb[phi_ar_comb<0].flatten().sum()    

phi_ar_comb_pos = phi_ar_comb[phi_ar_comb>=0].flatten().sum()
phi_ar_comb_neg = phi_ar_comb[phi_ar_comb<0].flatten().sum()    

hmi_ar_comb_unsigned = np.nansum(np.abs(hmi_ar_comb).flatten())
phi_ar_comb_unsigned = np.nansum(np.abs(phi_ar_comb).flatten())


In [ ]:
plt.close()
x = ['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative corr', 'HMI unsigned', 'PHI unsigned'] 
fig, ax = plt.subplots(figsize=(9,5))
#ax.bar(x, [hmi_ar_comb_pos, abs(hmi_ar_comb_neg), phi_ar_comb_off_pos, abs(phi_ar_comb_off_neg), hmi_ar_comb_unsigned, phi_ar_comb_off_unsigned], label='combined AR mask offset')
ax.bar(x, [hmi_ar_comb_pos, abs(hmi_ar_comb_neg), phi_ar_comb_pos, abs(phi_ar_comb_neg), hmi_ar_comb_unsigned, phi_ar_comb_unsigned], label='combined AR mask')
ax.bar(x, [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_ar_unsigned, phi_ar_unsigned], label='individual AR mask')

ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map /w combined AR mask") 

plt.legend()

In [ ]:
# Combined AR masks
ratio_unsigned_flux_comb = hmi_ar_comb_unsigned/phi_ar_comb_unsigned
print(f'Unsigned flux HMI {hmi_ar_comb_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_comb_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_comb:.4f}')

## HMI degradation with Airy Disk

Airy disk calculated from expected blur from missing FWHM

In [ ]:
from scipy.signal import fftconvolve
from astropy.convolution import convolve
from scipy.special import j1

In [ ]:
# Set up pixelscale info from cdelt keyword (currently manual!)
cdelt_hmi = 0.504032
cdelt_synop = 0.099963
cdelt_phi = 3.5748234469

In [ ]:
# todo check with function to use

In [ ]:
def airy_psf(size, radius_px):
    y, x = np.indices((size, size))
    r = np.sqrt((x - size//2)**2 + (y - size//2)**2)
    kr = np.pi * r / radius_px
    psf = np.ones_like(r)
    mask = kr != 0
    psf[mask] = (2 * j1(kr[mask]) / kr[mask])**2
    return psf / psf.sum()

In [ ]:
def airy_psf_v2(size, fwhm):
    """
    Create an Airy disk PSF.
    
    size: int, width/height of PSF kernel (pixels)
    fwhm: float, FWHM in pixels
    """
    y, x = np.indices((size, size)) - size // 2
    r = np.sqrt(x**2 + y**2)
    
    # Convert FWHM to first zero (approximation)
    r0 = fwhm / 1.028  # Airy FWHM ≈ 1.028 * lambda / D
    
    r_norm = np.pi * r / r0
    psf = (2 * j1(r_norm) / r_norm)**2
    psf[r==0] = 1.0  # handle division by zero at center
    psf /= psf.sum()
    return psf

In [ ]:
# Estimate FDT PSF from Airy disk

In [ ]:
theta_fdt = (1.22 * (617 * 1e-9)/0.0175)#*(24*3600)
sigma_fdt = theta_fdt / (2 * np.sqrt(2 * np.log(2)))  # Convert FWHM to sigma

theta_fdt_arcsec = np.degrees(theta_fdt)*(3600) 
sigma_fdt_arcsec = np.degrees(sigma_fdt)*(3600)
theta_fdt_arcsec, sigma_fdt_arcsec

In [ ]:
theta_hmi = (1.22 * (617 * 1e-9)/0.14)#*(24*3600)
sigma_hmi = theta_hmi / (2 * np.sqrt(2 * np.log(2)))  # Convert FWHM to sigma

theta_hmi_arcsec = np.degrees(theta_hmi)*(3600) 
sigma_hmi_arcsec = np.degrees(sigma_hmi)*(3600)

theta_hmi_arcsec, sigma_hmi_arcsec

In [ ]:
# 1800 px in remapped xdim. 3820 px solar disk in HMI: 1910 arcsec solar diameter / 0.504 arcsec/px = 3820 px
hmi_degr = 1800/3820.
print(f'HMI plate scale on reprojected/remapped frame {hmi_degr:.4f} arc/px') 

In [ ]:
fwhm_final = theta_fdt_arcsec/theta_hmi_arcsec
print(f'FWHM difference between HMI and PHI airy disks is a factor {fwhm_final}')

Find missing FWHM blur after HMI remapping process

In [ ]:
# FWHM_final = FWHM_existing + FWHM_kernel
fwhm_existing = hmi_degr * theta_fdt_arcsec/theta_hmi_arcsec
fwhm_kernel = np.sqrt(fwhm_final**2 - fwhm_existing**2)

print(f"Remaining blur as FWHM kernel size {fwhm_kernel:.2f} px")


In [ ]:
psf_to_fdt  = airy_psf_v2(65, fwhm_kernel)

In [ ]:
plt.close()
fig, ax1= plt.subplots(figsize=(12, 6), nrows=1, ncols=1, sharex=True, sharey=True)
im1 = ax1.imshow(psf_to_fdt)


In [ ]:
hmi_conv = convolve(hmi_img, psf_to_fdt, boundary='fill', fill_value=np.nan, normalize_kernel=True)


In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_img, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')
im2 = ax[1].imshow(hmi_conv, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI CR')

ax[0].set_title(f'PHI CR {carrington_number}')
ax[1].set_title(f'Convolved HMI CR {carrington_number} - degraded to FDT resolution')

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

HMI now seems too degraded compared to PHI. Why though? Will this look different if convolution is applied before reprojection step? This would be computationally VERY expensive.

In [ ]:
hmi_conv_ar_img, hmi_conv_ar_labels, hmi_conv_ar_mask = label_active_regions(hmi_conv, high_thld, low_thld, step)

In [ ]:
hmi_conv_ar_pos = hmi_conv_ar_img[hmi_conv_ar_img>=0].flatten().sum()
hmi_conv_ar_neg = hmi_conv_ar_img[hmi_conv_ar_img<0].flatten().sum()

hmi_conv_ar_unsigned = np.nansum(np.abs(hmi_conv_ar_img).flatten())

In [ ]:
plt.close()
x = ['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative corr', 'HMI unsigned', 'PHI unsigned'] 
fig, ax = plt.subplots(figsize=(9,5))
#ax.bar(x, [hmi_ar_comb_pos, abs(hmi_ar_comb_neg), phi_ar_comb_off_pos, abs(phi_ar_comb_off_neg), hmi_ar_comb_unsigned, phi_ar_comb_off_unsigned], label='combined AR mask offset')
#ax.bar(x, [hmi_ar_comb_pos, abs(hmi_ar_comb_neg), phi_ar_comb_pos, abs(phi_ar_comb_neg), hmi_ar_comb_unsigned, phi_ar_comb_unsigned], label='combined AR mask')
ax.bar(x, [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_ar_unsigned, phi_ar_unsigned], label='HMI fullres - individual AR mask')
#ax.bar(x, [hmi_conv_ar_pos, abs(hmi_conv_ar_neg), phi_ar_comb_pos, abs(phi_ar_comb_neg), hmi_conv_ar_unsigned, phi_ar_comb_unsigned], label='HMI degraded - individual AR mask')
ax.bar(x, [hmi_conv_ar_pos, abs(hmi_conv_ar_neg), phi_pos, abs(phi_neg), hmi_conv_ar_unsigned, phi_ar_unsigned], label='HMI degraded - individual AR mask')

ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map /w combined AR mask") 

plt.legend()

In [ ]:
# HMI degraded, individual AR masks
ratio_unsigned_flux_conv = hmi_conv_ar_unsigned/phi_ar_unsigned
print(f'Unsigned flux HMI {hmi_conv_ar_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_conv:.4f}')

## HMI degradation with Airy Disk & combined AR mask

In [ ]:
# create master mask from phi_ar_mask and hmi_conv_ar_mask
ar_mask2 = hmi_conv_ar_mask | phi_ar_mask

In [ ]:
phi_ar_comb2  = np.where(ar_mask2, phi_img,  np.nan)
hmi_ar_comb2  = np.where(ar_mask2, hmi_conv, np.nan)

In [ ]:
phi_ar_comb2_pos = phi_ar_comb2[phi_ar_comb2>=0].flatten().sum()
phi_ar_comb2_neg = phi_ar_comb2[phi_ar_comb2<0].flatten().sum()

phi_ar_comb2_unsigned = np.nansum(np.abs(phi_ar_comb2).flatten())

hmi_ar_comb2_pos = hmi_ar_comb2[hmi_ar_comb2>=0].flatten().sum()
hmi_ar_comb2_neg = hmi_ar_comb2[hmi_ar_comb2<0].flatten().sum()

hmi_ar_comb2_unsigned = np.nansum(np.abs(hmi_ar_comb2).flatten())

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 12), nrows=3, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar_comb2, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_ar_comb2, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='HMI')
im3 = ax[2].imshow(hmi_conv_ar_img,  vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='HMI')

ax[0].set_title(f'PHI Active Regions CR {carrington_number} - combined mask')
ax[1].set_title(f'HMI Active Regions CR {carrington_number} - degraded, combined AR mask')
ax[2].set_title(f'HMI Active Regions CR {carrington_number} - degraded, individual AR mask')

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')
ax[2].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')
fig.colorbar(im3, label='Br [G]')

In [ ]:
plt.close()
x = ['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative corr', 'HMI unsigned', 'PHI unsigned'] 
fig, ax = plt.subplots(figsize=(9,5))
#ax.bar(x, [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_ar_unsigned, phi_ar_unsigned], label='HMI fullres - individual AR mask')
ax.bar(x, [hmi_ar_comb2_pos, abs(hmi_ar_comb2_neg), phi_ar_comb2_pos, abs(phi_ar_comb2_neg), hmi_ar_comb2_unsigned, phi_ar_comb2_unsigned], label='HMI degraded - combined AR mask')
ax.bar(x, [hmi_conv_ar_pos, abs(hmi_conv_ar_neg), phi_pos, abs(phi_neg), hmi_conv_ar_unsigned, phi_ar_unsigned], label='HMI degraded - individual AR mask')

ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map /w combined AR mask") 

plt.legend()

In [ ]:
# HMI degraded, combined AR masks
ratio_unsigned_flux_comb2 = hmi_ar_comb2_unsigned/phi_ar_comb_unsigned
print(f'Unsigned flux HMI {hmi_ar_comb2_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_comb_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_comb2:.4f}')

## Small Synoptic Test

In [ ]:
phi_ar_comb2_small = phi_ar_comb2.reshape(360, 4, 720, 5).mean(axis=(1, 3))
hmi_ar_comb2_small = hmi_ar_comb2.reshape(360, 4, 720, 5).mean(axis=(1, 3))

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(6,6), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar_comb2_small, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_ar_comb2_small, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

In [ ]:
phi_ar_comb2_small_pos = phi_ar_comb2_small[phi_ar_comb2_small>=0].flatten().sum()
phi_ar_comb2_small_neg = phi_ar_comb2_small[phi_ar_comb2_small<0].flatten().sum()

phi_ar_comb2_small_unsigned = np.nansum(np.abs(phi_ar_comb2_small).flatten())

hmi_ar_comb2_small_pos = hmi_ar_comb2_small[hmi_ar_comb2_small>=0].flatten().sum()
hmi_ar_comb2_small_neg = hmi_ar_comb2_small[hmi_ar_comb2_small<0].flatten().sum()

hmi_ar_comb2_small_unsigned = np.nansum(np.abs(hmi_ar_comb2_small).flatten())

In [ ]:
plt.close()
x = ['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative corr', 'HMI unsigned', 'PHI unsigned'] 
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(x, [hmi_ar_comb2_pos, abs(hmi_ar_comb2_neg), phi_ar_comb2_pos, abs(phi_ar_comb2_neg), hmi_ar_comb2_unsigned, phi_ar_comb2_unsigned], label='HMI degraded - combined AR mask')
ax.bar(x, [hmi_ar_comb2_small_pos, abs(hmi_ar_comb2_small_neg), phi_ar_comb2_small_pos, abs(phi_ar_comb2_small_neg), hmi_ar_comb2_small_unsigned, phi_ar_comb2_small_unsigned], label='HMI degraded - combined AR mask - small synoptic')


ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map /w combined AR mask") 

plt.legend()

In [ ]:
# HMI degraded, combined AR masks
ratio_unsigned_flux_comb2_small = hmi_ar_comb2_small_unsigned/phi_ar_comb2_small_unsigned
print(f'Unsigned flux HMI {hmi_ar_comb2_small_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_comb2_small_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_comb2_small:.4f}')

## Force flux balance with PHI offset

In [ ]:
hmi_flux_ratio = hmi_pos/abs(hmi_neg)
phi_flux_ratio = phi_pos/abs(phi_neg)

print(f'Flux balance at full resolution')
print(f'HMI flux ratio: {hmi_flux_ratio:.4f}')
print(f'PHI flux ratio: {phi_flux_ratio:.4f}')

# HMI slightly positive but somewhat balanced, PHI a skewed to negative

In [ ]:
hmi_comb2_flux_ratio = hmi_ar_comb2_pos/abs(hmi_ar_comb2_neg)
phi_comb2_flux_ratio = phi_ar_comb2_pos/abs(phi_ar_comb2_neg)

print(f'Flux balance at at degraded resolution')
print(f'HMI flux ratio: {hmi_comb2_flux_ratio:.4f}')
print(f'PHI flux ratio: {phi_comb2_flux_ratio:.4f}')

# Both PHI and HMI are now negatively skewed. PHI more balanced than HMI


In [ ]:
offset1 = 2.5
phi_ar_off = phi_ar_img + offset1

phi_ar_off_pos = phi_ar_off[phi_ar_off>=0].flatten().sum()
phi_ar_off_neg = phi_ar_off[phi_ar_off<0].flatten().sum()    

phi_ar_off_unsigned = np.nansum(np.abs(phi_ar_off).flatten())


In [ ]:
plt.close()
x = ['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative corr', 'HMI unsigned', 'PHI unsigned'] 
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(x, [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_ar_unsigned, phi_ar_unsigned], label='HMI fullres - individual AR mask')

ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map /w individual AR mask") 

plt.legend()


In [ ]:
plt.close()
x = ['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative corr', 'HMI unsigned', 'PHI unsigned'] 
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(x, [hmi_pos, abs(hmi_neg), phi_ar_off_pos, abs(phi_ar_off_neg), hmi_ar_unsigned, phi_ar_off_unsigned], label='HMI fullres - individual AR mask - PHI offset')
#ax.bar(x, [hmi_ar_comb2_pos, abs(hmi_ar_comb2_neg), phi_ar_comb2_off_pos, abs(phi_ar_comb2_off_neg), hmi_ar_comb2_unsigned, phi_ar_comb2_off_unsigned], label='combined AR mask offset')

ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map /w combined AR mask") 

plt.legend()


In [ ]:
# Flux balance original resolution and individual AR masks

hmi_flux_ratio = hmi_pos/abs(hmi_neg)
phi_flux_ratio = phi_pos/abs(phi_neg)

print(f'Flux balance at full resolution')
print(f'HMI flux ratio: {hmi_flux_ratio:.4f}')
print(f'PHI flux ratio: {phi_flux_ratio:.4f}')
print()

# HMI slightly positive but somewhat balanced, PHI a skewed to negative
phi_flux_ratio_off = phi_ar_off_pos/abs(phi_ar_off_neg)
print(f'Flux balance for PHI offset: {offset1:.2f} G')
print(f'PHI flux ratio w/ offset: {phi_flux_ratio_off:.4f}')


In [ ]:
offset2 = 1.5
phi_ar_comb2_off = phi_ar_comb2 + offset2

phi_ar_comb2_off_pos = phi_ar_comb2_off[phi_ar_comb2_off>=0].flatten().sum()
phi_ar_comb2_off_neg = phi_ar_comb2_off[phi_ar_comb2_off<0].flatten().sum()    

phi_ar_comb2_off_unsigned = np.nansum(np.abs(phi_ar_comb2_off).flatten())


In [ ]:
plt.close()
x = ['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative corr', 'HMI unsigned', 'PHI unsigned'] 
fig, ax = plt.subplots(figsize=(9,5))
#ax.bar(x, [hmi_pos, abs(hmi_neg), phi_ar_off_pos, abs(phi_ar_off_neg), hmi_ar_unsigned, phi_ar_off_unsigned], label='HMI fullres - individual AR mask')
ax.bar(x, [hmi_ar_comb2_pos, abs(hmi_ar_comb2_neg), phi_ar_comb2_off_pos, abs(phi_ar_comb2_off_neg), hmi_ar_comb2_unsigned, phi_ar_comb2_off_unsigned], label='combined AR mask offset')

ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map /w combined AR mask") 

plt.legend()


In [ ]:
# Flux balance with HMI degraded to FDT resolution and combined AR mask

hmi_comb2_flux_ratio = hmi_ar_comb2_pos/abs(hmi_ar_comb2_neg)
phi_comb2_off_flux_ratio = phi_ar_comb2_off_pos/abs(phi_ar_comb2_off_neg)

print(f'Flux balance at at degraded resolution')
print(f'HMI flux ratio: {hmi_comb2_flux_ratio:.4f}')
print(f'PHI flux ratio: {phi_comb2_flux_ratio:.4f}')
print()

# Both PHI and HMI are now negatively skewed. PHI more balanced than HMI
print(f'Flux balance for PHI offset: {offset2:.2f} G')
print(f'PHI flux ratio w/offset: {phi_comb2_off_flux_ratio:.4f}')


In [ ]:
# HMI degraded, PHI offset corrected, combined AR masks
ratio_unsigned_flux_comb2_off = hmi_ar_comb2_unsigned/phi_ar_comb2_off_unsigned
print(f'Unsigned flux HMI {hmi_ar_comb2_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_comb2_unsigned:.0f} Mx')
print(f'Unsigned flux PHI offset corrected {phi_ar_comb2_off_unsigned:.0f} Mx')
print()

print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_comb2:.4f}')
print(f'Ratio unsigned flux HMI/PHI offset corrected {ratio_unsigned_flux_comb2_off:.4f}')

# offset correction makes HMI/PHI ratio slightly worse

## Cross Calibration Density Plots

In [ ]:
bins = 400
vmin = -2000
vmax = 2000

fig, (ax1, ax2)= plt.subplots(figsize=(14, 6), nrows=1, ncols=2, sharex=True, sharey=True)
counts1, xedges1, yedges1, im1 = ax1.hist2d(phi_ar_comb2.flatten(), hmi_ar_comb2.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")
counts2, xedges2, yedges2, im2 = ax2.hist2d(phi_ar_comb.flatten(), hmi_ar_comb.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")

ax1.set_xlim(vmin, vmax)
ax1.set_ylim(vmin, vmax)

ax2.set_xlim(vmin, vmax)
ax2.set_ylim(vmin, vmax)

ax1.set_xlabel("PHI Magnetic Field [G]")
ax1.set_ylabel("HMI Magnetic Field [G]")

ax2.set_xlabel("PHI Magnetic Field [G]")
ax2.set_ylabel("HMI Magnetic Field [G]")

ax1.set_title("HMI degraded, combined mask")
ax2.set_title("HMI full resolution, combined mask")

counts, xedges, yedges = counts1, xedges1, yedges1

xcenters = 0.5 * (xedges[:-1] + xedges[1:])
ycenters = 0.5 * (yedges[:-1] + yedges[1:])

X, Y = np.meshgrid(xcenters, ycenters, indexing="ij")
Z = counts

mask = Z > 0

xdata = X[mask]
ydata = Y[mask]
weights = Z[mask]

m, b = np.polyfit(xdata, ydata, 1, w=weights)

print(f"slope = {m}")
print(f"intercept = {b}")

##################

xx = np.linspace(vmin, vmax, 500)
yy = m* xx + b
ax1.plot(xx, yy, color="red", lw=2)


## Summary

In [ ]:
# Individual AR masks
ratio_unsigned_flux = hmi_ar_unsigned/phi_ar_unsigned
print(f'Unsigned flux HMI {hmi_ar_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux:.4f}')

In [ ]:
# Combined AR masks
ratio_unsigned_flux_comb = hmi_ar_comb_unsigned/phi_ar_comb_unsigned
print(f'Unsigned flux HMI {hmi_ar_comb_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_comb_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_comb:.4f}')

In [ ]:
# HMI degraded, individual AR masks
ratio_unsigned_flux_conv = hmi_conv_ar_unsigned/phi_ar_unsigned
print(f'Unsigned flux HMI {hmi_conv_ar_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_conv:.4f}')

In [ ]:
# HMI degraded, combined AR masks
ratio_unsigned_flux_comb2 = hmi_ar_comb2_unsigned/phi_ar_comb_unsigned
print(f'Unsigned flux HMI {hmi_ar_comb2_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_comb_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_comb2:.4f}')

In [ ]:
# HMI degraded, combined AR masks, small synoptic
ratio_unsigned_flux_comb2_small = hmi_ar_comb2_small_unsigned/phi_ar_comb2_small_unsigned
print(f'Unsigned flux HMI {hmi_ar_comb2_small_unsigned:.0f} Mx')
print(f'Unsigned flux PHI {phi_ar_comb2_small_unsigned:.0f} Mx')
print(f'Ratio unsigned flux HMI/PHI {ratio_unsigned_flux_comb2_small:.4f}')

In [ ]:
# Flux balance original resolution and individual AR masks

hmi_flux_ratio = hmi_pos/abs(hmi_neg)
phi_flux_ratio = phi_pos/abs(phi_neg)

print(f'Flux balance at full resolution')
print(f'HMI flux ratio: {hmi_flux_ratio:.4f}')
print(f'PHI flux ratio: {phi_flux_ratio:.4f}')
print()

# HMI slightly positive but somewhat balanced, PHI a skewed to negative
phi_flux_ratio_off = phi_ar_off_pos/abs(phi_ar_off_neg)
print(f'Flux balance for PHI offset: {offset1:.2f} G')
print(f'PHI flux ratio w/ offset: {phi_flux_ratio_off:.4f}')


In [ ]:
# Flux balance with HMI degraded to FDT resolution and combined AR mask

hmi_comb2_flux_ratio = hmi_ar_comb2_pos/abs(hmi_ar_comb2_neg)
phi_comb2_flux_ratio = phi_ar_comb2_pos/abs(phi_ar_comb2_neg)

print(f'Flux balance at at degraded resolution')
print(f'HMI flux ratio: {hmi_comb2_flux_ratio:.4f}')
print(f'PHI flux ratio: {phi_comb2_flux_ratio:.4f}')
print()

# Both PHI and HMI are now negatively skewed. PHI more balanced than HMI
phi_comb2_flux_ratio_off = phi_ar_comb2_off_pos/abs(phi_ar_comb2_off_neg)
print(f'Flux balance for PHI offset: {offset2:.2f} G')
print(f'PHI flux ratio w/offset: {phi_comb2_flux_ratio_off:.4f}')


## Region Labels

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar_labels, interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_ar_labels, interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)


fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_ar_img, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_ar_img, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

Flux ratio unchanged by additional surface, but balanced by adding 2.5G offset to the entire map

## Active Region Window

In [ ]:
# Select data segment
x1 = 900
x2 = 1500
y1 = 750
y2 = 1100

hmi_seg = hmi_ar_comb2[y1:y2, x1:x2] # hmi standard synoptic
phi_seg = phi_ar_comb2[y1:y2, x1:x2] # hmi standard synoptic

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(6,6), nrows=2, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_seg, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_seg, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')

In [ ]:
from matplotlib.colors import LogNorm

####### DENSITY PLOTS ########

vmin = -2000
vmax = +2000

bins = 200

fig, ax = plt.subplots(figsize=(6,5))   
counts, xedges, yedges, im = ax.hist2d(phi_seg.flatten(), hmi_seg.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")

xcenters = 0.5 * (xedges[:-1] + xedges[1:])
ycenters = 0.5 * (yedges[:-1] + yedges[1:])

X, Y = np.meshgrid(xcenters, ycenters, indexing="ij")
Z = counts

mask = Z > 0

xdata = X[mask]
ydata = Y[mask]
weights = Z[mask]

m, b = np.polyfit(xdata, ydata, 1, w=weights)

print(f"slope = {m}")
print(f"intercept = {b}")

##################

xx = np.linspace(vmin, vmax, 500)
yy = m* xx + b
ax.plot(xx, yy, color="red", lw=2)

In [ ]:
# This result is close to Alejandros cross calibratin!

# Playground

In [ ]:
tpath = "/scratch/slam/loeschl/dev/python/synoptic-map-pipeline/output/CR2291_PHI_only_blos/data/solo_L2_phi-fdt-blos_20241112T001503_V202509221936_0451120501_drms.fits"

test = fits.open(tpath)

In [ ]:
test[1].header

## Airy Disk Convolution

In [ ]:
cdelt_hmi = 0.504032
cdelt_synop = 0.099963
cdelt_phi = 3.5748234469

In [ ]:
from scipy.signal import fftconvolve
from astropy.convolution import convolve
from scipy.special import j1

In [ ]:
def airy_psf(size, radius_px):
    y, x = np.indices((size, size))
    r = np.sqrt((x - size//2)**2 + (y - size//2)**2)
    kr = np.pi * r / radius_px
    psf = np.ones_like(r)
    mask = kr != 0
    psf[mask] = (2 * j1(kr[mask]) / kr[mask])**2
    return psf / psf.sum()

In [ ]:
def airy_psf_v2(size, fwhm):
    """
    Create an Airy disk PSF.
    
    size: int, width/height of PSF kernel (pixels)
    fwhm: float, FWHM in pixels
    """
    y, x = np.indices((size, size)) - size // 2
    r = np.sqrt(x**2 + y**2)
    
    # Convert FWHM to first zero (approximation)
    r0 = fwhm / 1.028  # Airy FWHM ≈ 1.028 * lambda / D
    
    r_norm = np.pi * r / r0
    psf = (2 * j1(r_norm) / r_norm)**2
    psf[r==0] = 1.0  # handle division by zero at center
    psf /= psf.sum()
    return psf

In [ ]:
# Estimate FDT PSF from Airy disk

In [ ]:
theta_fdt = (1.22 * (617 * 1e-9)/0.0175)#*(24*3600)
sigma_fdt = theta_fdt / (2 * np.sqrt(2 * np.log(2)))  # Convert FWHM to sigma

theta_fdt_arcsec = np.degrees(theta_fdt)*(3600) 
sigma_fdt_arcsec = np.degrees(sigma_fdt)*(3600)
theta_fdt_arcsec, sigma_fdt_arcsec

In [ ]:
theta_hmi = (1.22 * (617 * 1e-9)/0.14)#*(24*3600)
sigma_hmi = theta_hmi / (2 * np.sqrt(2 * np.log(2)))  # Convert FWHM to sigma

theta_hmi_arcsec = np.degrees(theta_hmi)*(3600) 
sigma_hmi_arcsec = np.degrees(sigma_hmi)*(3600)

theta_hmi_arcsec, sigma_hmi_arcsec

In [ ]:
# 1800 px in remapped xdim. 3820 px solar disk in HMI: 1910 arcsec solar diameter / 0.504 arcsec/px = 3820 px
hmi_degr = 1800/3820.
hmi_degr

In [ ]:
fwhm_final = theta_fdt_arcsec/theta_hmi_arcsec
fwhm_final

In [ ]:
# FWHM_final = FWHM_existing + FWHM_kernel
fwhm_existing = hmi_degr * theta_fdt_arcsec/theta_hmi_arcsec
fwhm_kernel = np.sqrt(fwhm_final**2 - fwhm_existing**2)
fwhm_kernel


In [ ]:
radius_px = fwhm_existing

In [ ]:
# radius_px = (1.22 * wavelength / aperture) / pixel_scale 
radius_px_fdt = theta_fdt_arcsec / cdelt_phi
radius_px_hmi = theta_hmi_arcsec / cdelt_hmi

radius_px_fdt, radius_px_hmi

In [ ]:
psf_fdt = airy_psf(size=31, radius_px=radius_px_fdt)

In [ ]:
def airy_psf_v2(size, fwhm):
    """
    Create an Airy disk PSF.
    
    size: int, width/height of PSF kernel (pixels)
    fwhm: float, FWHM in pixels
    """
    y, x = np.indices((size, size)) - size // 2
    r = np.sqrt(x**2 + y**2)
    
    # Convert FWHM to first zero (approximation)
    r0 = fwhm / 1.028  # Airy FWHM ≈ 1.028 * lambda / D
    
    r_norm = np.pi * r / r0
    psf = (2 * j1(r_norm) / r_norm)**2
    psf[r==0] = 1.0  # handle division by zero at center
    psf /= psf.sum()
    return psf

# Example usage:

#fwhm_map_pixels = radius_px_fdt
#psf_fdt2 = airy_psf_v2(int(3*fwhm_map_pixels), fwhm_map_pixels)
psf_fdt  = airy_psf_v2(65, fwhm_kernel)
psf_fdt2 = airy_psf_v2(65, 2* radius_px_fdt)


In [ ]:
radius_px_fdt, 2*radius_px_fdt

In [ ]:
plt.close()
fig, (ax1, ax2)= plt.subplots(figsize=(12, 6), nrows=1, ncols=2, sharex=True, sharey=True)
im1 = ax1.imshow(psf_fdt )
im2 = ax2.imshow(psf_fdt2)

In [ ]:
hmi_conv  = convolve(hmi_img, psf_fdt, boundary='fill', fill_value=np.nan, normalize_kernel=True)
hmi_conv2 = convolve(hmi_img, psf_fdt2, boundary='fill', fill_value=np.nan, normalize_kernel=True)

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 8), nrows=3, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_img, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='PHI CR')
im2 = ax[1].imshow(hmi_conv, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI CR')
im3 = ax[2].imshow(hmi_conv2, vmin=-1500, vmax=1500, cmap="hmimag", interpolation='None', origin='lower', label='HMI CR')

ax[0].set_title('PHI CR %s'%carrington_number)
ax[1].set_title('Convolved HMI CR %s'%carrington_number)
ax[2].set_title('Convolved HMI CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')
ax[2].set_facecolor('lightgray')
fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')
fig.colorbar(im3, label='Br [G]')

In [ ]:
phi_ar_img, phi_ar_labels = label_active_regions(phi_img, high_thld, low_thld, step)
hmi_ar_img, hmi_ar_labels = label_active_regions(hmi_img, high_thld, low_thld, step)
hmi_conv_ar_img, hmi_conv_ar_labels = label_active_regions(hmi_conv, high_thld, low_thld, step)

In [ ]:
hmi_pos = hmi_ar_img[hmi_ar_img>=0].flatten().sum()
hmi_neg = hmi_ar_img[hmi_ar_img<0].flatten().sum()

hmi_conv_pos = hmi_conv_ar_img[hmi_conv_ar_img>=0].flatten().sum()
hmi_conv_neg = hmi_conv_ar_img[hmi_conv_ar_img<0].flatten().sum()

phi_pos = phi_ar_img[phi_ar_img>=0].flatten().sum()
phi_neg = phi_ar_img[phi_ar_img<0].flatten().sum()    

hmi_ar_unsigned = np.nansum(np.abs(hmi_ar_img).flatten())
phi_ar_unsigned = np.nansum(np.abs(phi_ar_img).flatten())
hmi_conv_ar_unsigned = np.nansum(np.abs(hmi_conv_ar_img).flatten())

In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative', 'HMI unsigned', 'PHI unsigned'], [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_ar_unsigned, phi_ar_unsigned])
ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map AR") 

In [ ]:
# build combined mask from HMI and PHI AR masks
hmi_ar_mask = ~np.isnan(hmi_ar_img)
phi_ar_mask = ~np.isnan(phi_ar_img)
hmi_conv_ar_mask = ~np.isnan(hmi_conv_ar_img)

#ar_mask = np.logical_or.reduce([hmi_ar_mask, phi_ar_mask, hmi_conv_ar_mask])
ar_mask = np.logical_or.reduce([phi_ar_mask, hmi_conv_ar_mask])


len(hmi_ar_mask[hmi_ar_mask==True]), len(phi_ar_mask[phi_ar_mask==True]), len(hmi_conv_ar_mask[hmi_conv_ar_mask==True]), len(ar_mask[ar_mask==True])

In [ ]:
phi_master  = np.where(ar_mask, phi_img, np.nan)
hmi_master  = np.where(ar_mask, hmi_img, np.nan)
hmi_conv_master = np.where(ar_mask, hmi_conv, np.nan)


In [ ]:
plt.close()
fig, ax = plt.subplots(figsize=(10, 12), nrows=3, ncols=1, sharex=True, sharey=True)
im1 = ax[0].imshow(phi_master, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='PHI')
im2 = ax[1].imshow(hmi_conv_master, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='HMI')
im3 = ax[2].imshow(hmi_master, vmin=-1500, vmax=1500, cmap='hmimag', interpolation='None', origin='lower', label='HMI')

ax[0].set_title('PHI Active Regions CR %s'%carrington_number)
ax[1].set_title('HMI Conv Active Regions CR %s'%carrington_number)
ax[2].set_title('HMI Active Regions CR %s'%carrington_number)

ax[0].set_facecolor('lightgray')
ax[1].set_facecolor('lightgray')
ax[2].set_facecolor('lightgray')

fig.colorbar(im1, label='Br [G]')
fig.colorbar(im2, label='Br [G]')
fig.colorbar(im3, label='Br [G]')

In [ ]:

hmi_master_pos = hmi_master[hmi_master>=0].flatten().sum()
hmi_master_neg = hmi_master[hmi_master<0].flatten().sum()

phi_master_pos = phi_master[phi_master>=0].flatten().sum()
phi_master_neg = phi_master[phi_master<0].flatten().sum()    

hmi_conv_master_pos = hmi_conv_master[hmi_conv_master>=0].flatten().sum()
hmi_conv_master_neg = hmi_conv_master[hmi_conv_master<0].flatten().sum()    

hmi_master_unsigned = np.nansum(np.abs(hmi_master).flatten())
phi_master_unsigned = np.nansum(np.abs(phi_master).flatten())
hmi_conv_master_unsigned = np.nansum(np.abs(hmi_conv_master).flatten())

# should now be the same length due to combined mask
n_phi_master = len(phi_master[~np.isnan(phi_master)])
n_hmi_master = len(hmi_master[~np.isnan(hmi_master)])
n_hmi_conv_master = len(hmi_conv_master[~np.isnan(hmi_conv_master)])

In [ ]:
hmi_conv2_master = np.where(ar_mask, hmi_conv2, np.nan)
hmi_conv2_master_pos = hmi_conv2_master[hmi_conv2_master>=0].flatten().sum()
hmi_conv2_master_neg = hmi_conv2_master[hmi_conv2_master<0].flatten().sum()    
hmi_conv2_master_unsigned = np.nansum(np.abs(hmi_conv2_master).flatten())

In [ ]:
n_phi_ar_img = len(phi_ar_img[~np.isnan(phi_ar_img)])
n_hmi_ar_img = len(hmi_ar_img[~np.isnan(hmi_ar_img)])
n_hmi_conv_ar_img = len(hmi_conv_ar_img[~np.isnan(hmi_conv_ar_img)])   

In [ ]:
offset = 1.7

phi_master_off = phi_master + offset

phi_master_off_pos = phi_master_off[phi_master_off>=0].flatten().sum()
phi_master_off_neg = phi_master_off[phi_master_off<0].flatten().sum()    

phi_master_off_pos = phi_master_off[phi_master_off>=0].flatten().sum()
phi_master_off_neg = phi_master_off[phi_master_off<0].flatten().sum()    

phi_master_off_unsigned = np.nansum(np.abs(phi_master_off).flatten())


In [ ]:
phi_master_off_pos/phi_master_off_neg, hmi_master_pos/hmi_master_neg, hmi_conv_master_pos/hmi_conv_master_neg

In [ ]:
plt.close()
x = ['HMI Positive', 'HMI Negative', 'PHI Positive', 'PHI Negative corr', 'HMI unsigned', 'PHI unsigned'] 
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(x, [hmi_master_pos, abs(hmi_master_neg), phi_master_pos, abs(phi_master_neg), hmi_master_unsigned, phi_master_unsigned], label='combined AR mask')
#ax.bar(x, [hmi_pos, abs(hmi_neg), phi_pos, abs(phi_neg), hmi_ar_unsigned, phi_ar_unsigned], label='individual AR mask')

#ax.bar(x, [hmi_conv2_master_pos, abs(hmi_conv2_master_neg), phi_master_pos, abs(phi_master_neg), hmi_conv2_master_unsigned, phi_master_unsigned], label='master mask / convolution')
#ax.bar(x, [hmi_conv2_master_pos, abs(hmi_conv2_master_neg), phi_master_off_pos, abs(phi_master_off_neg), hmi_conv2_master_unsigned, phi_master_off_unsigned], label='master mask / convolution/ offset corr')

#ax.bar(x, [hmi_conv_master_pos, abs(hmi_conv_master_neg), phi_master_pos, abs(phi_master_neg), hmi_conv_master_unsigned, phi_master_unsigned], label='master mask / convolution')
ax.bar(x, [hmi_conv_master_pos, abs(hmi_conv_master_neg), phi_master_off_pos, abs(phi_master_off_neg), hmi_conv_master_unsigned, phi_master_off_unsigned], label='master mask / convolution/ offset corr')


#ax.bar(x, [hmi_conv_pos, abs(hmi_conv_neg), phi_pos, abs(phi_neg), hmi_conv_ar_unsigned, phi_ar_unsigned], label='HMI mask / convolution')

ax.set_ylabel("Magnetic Flux [Mx]")
ax.set_title("Magnetic Flux Comparison between HMI and PHI Synoptic Map AR combined mask") 

plt.legend()

In [ ]:
phi_master_pos/phi_master_neg, phi_master_off_pos/phi_master_off_neg

In [ ]:
ratio_conv_master = hmi_conv_master_unsigned/phi_master_unsigned
ratio_conv_ar     = hmi_conv_ar_unsigned/phi_ar_unsigned
ratio_ar          = hmi_ar_unsigned/phi_ar_unsigned 
ratio_conv_master, ratio_conv_ar, ratio_ar

In [ ]:
bins = 400
vmin = -2000
vmax = 2000

fig, (ax1, ax2)= plt.subplots(figsize=(14, 6), nrows=1, ncols=2, sharex=True, sharey=True)
counts1, xedges1, yedges1, im1 = ax1.hist2d(phi_master.flatten(), hmi_conv_master.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")
counts2, xedges2, yedges2, im2 = ax2.hist2d(phi_img.flatten(), hmi_img.flatten(), bins=(bins, bins), range=((vmin, vmax), (vmin, vmax)), norm=LogNorm(), cmap="cividis")

ax1.set_xlim(vmin, vmax)
ax1.set_ylim(vmin, vmax)
ax2.set_xlim(vmin, vmax)
ax2.set_ylim(vmin, vmax)

ax1.set_xlabel("PHI Magnetic Field [G]")
ax1.set_ylabel("HMI Magnetic Field [G]")

ax2.set_xlabel("PHI Magnetic Field [G]")
ax2.set_ylabel("HMI Magnetic Field [G]")


counts, xedges, yedges = counts1, xedges1, yedges1

xcenters = 0.5 * (xedges[:-1] + xedges[1:])
ycenters = 0.5 * (yedges[:-1] + yedges[1:])

X, Y = np.meshgrid(xcenters, ycenters, indexing="ij")
Z = counts

mask = Z > 0

xdata = X[mask]
ydata = Y[mask]
weights = Z[mask]

m, b = np.polyfit(xdata, ydata, 1, w=weights)

print(f"slope = {m}")
print(f"intercept = {b}")

##################


xx = np.linspace(vmin, vmax, 500)
yy = m* xx + b
ax1.plot(xx, yy, color="red", lw=2)



